In [1]:
# 1. 라이브러리 불러오기 및 실습 데이터 생성
# 실습을 위해 12명 고객의 연속형 나이 데이터를 생성.

import pandas as pd
import numpy as np

# 실습용 데이터 생성
data = {
    '고객': [f'고객_{i}' for i in range(1, 13)],
    '나이': [15, 22, 25, 29, 31, 33, 37, 44, 48, 52, 55, 68]
}
df = pd.DataFrame(data)
print("--- 원본 데이터 (연속형 수치) ---")
print(df.T) # 가독성을 위해 가로로 출력

--- 원본 데이터 (연속형 수치) ---
      0     1     2     3     4     5     6     7     8      9      10     11
고객  고객_1  고객_2  고객_3  고객_4  고객_5  고객_6  고객_7  고객_8  고객_9  고객_10  고객_11  고객_12
나이    15    22    25    29    31    33    37    44    48     52     55     68


In [3]:
# 2. pd.cut() — 동일 길이 분할 (Equal-width binning)
# •	원리: 데이터의 최소값부터 최대값까지의 구간 거리를 일정하게 3등분.
# •	특징: 각 구간의 길이는 똑같지만, 특정 구간에 데이터가 몰려있다면 구간별 데이터 개수는 제각각일 수 있.

# 방법 A: 개수 지정형 (알아서 3등분)

df_cut = df.copy()

# 나이 범위를 바탕으로 동일한 길이의 구간 3개 생성
df_cut['나이구간_길이형'] = pd.cut(df_cut['나이'], bins=3)

print("--- 1. pd.cut() 결과 (구간 범위 확인) ---")
print(df_cut[['고객', '나이', '나이구간_길이형']])

# 출력 결과 설명: 구간이 (14.947, 32.667], (32.667, 50.333], (50.333, 68.0] 와 같이 소수점으로 정밀하게 균등 배분됩니다.

--- 1. pd.cut() 결과 (구간 범위 확인) ---
       고객  나이          나이구간_길이형
0    고객_1  15  (14.947, 32.667]
1    고객_2  22  (14.947, 32.667]
2    고객_3  25  (14.947, 32.667]
3    고객_4  29  (14.947, 32.667]
4    고객_5  31  (14.947, 32.667]
5    고객_6  33  (32.667, 50.333]
6    고객_7  37  (32.667, 50.333]
7    고객_8  44  (32.667, 50.333]
8    고객_9  48  (32.667, 50.333]
9   고객_10  52    (50.333, 68.0]
10  고객_11  55    (50.333, 68.0]
11  고객_12  68    (50.333, 68.0]


In [4]:
# 방법 B: 경계선 직접 지정 및 라벨링 (실무 추천)
# 소수점 구간은 보기 불편하므로 명확한 경계 수치와 이름(labels)을 부여.

# 경계선 설정 (10~30세, 30~50세, 50~70세)
# bins 리스트의 원소 개수가 4개이면 3개의 구간이 만들어집니다.
bins_edges = [10, 30, 50, 70]
group_names = ['청년층', '중년층', '장년층']

df_cut['연령대_라벨'] = pd.cut(df_cut['나이'], bins=bins_edges, labels=group_names)

print("\n--- 2. pd.cut() 사용자 지정 라벨 적용 ---")
print(df_cut[['고객', '나이', '연령대_라벨']])


--- 2. pd.cut() 사용자 지정 라벨 적용 ---
       고객  나이 연령대_라벨
0    고객_1  15    청년층
1    고객_2  22    청년층
2    고객_3  25    청년층
3    고객_4  29    청년층
4    고객_5  31    중년층
5    고객_6  33    중년층
6    고객_7  37    중년층
7    고객_8  44    중년층
8    고객_9  48    중년층
9   고객_10  52    장년층
10  고객_11  55    장년층
11  고객_12  68    장년층


In [5]:
# 3. pd.qcut() — 동일 개수 분할 (Equal-frequency binning)
# •	원리: 데이터의 분포를 고려하여 각 구간에 들어가는 데이터의 개수가 똑같도록 등분. (사분위수 기반)
# •	특징: 구간의 길이는 제각각이 되지만, 각 범주에 속한 데이터 샘플 수가 균등하게 배정되므로 데이터 불균형 문제를 예방할 수 있다.

df_qcut = df.copy()

# q=3 으로 지정하면 전체 12개 데이터를 4개씩 쪼개어 3개의 구간을 만듭니다.
df_qcut['나이구간_개수형'] = pd.qcut(df_qcut['나이'], q=3)

print("--- 3. pd.qcut() 결과 (구간당 샘플 개수 균등) ---")
print(df_qcut[['고객', '나이', '나이구간_개수형']])

print("\n[구간별 데이터 개수 확인]")
print(df_qcut['나이구간_개수형'].value_counts())

# 출력 결과 설명: 데이터가 4개씩 정확히 삼등분되어 각 범위의 길이는 분포에 따라 다르게 설정됩니다. qcut 역시 labels 옵션을 지원하므로 원하시면 똑같이 이름을 붙일 수 있다.

--- 3. pd.qcut() 결과 (구간당 샘플 개수 균등) ---
       고객  나이          나이구간_개수형
0    고객_1  15  (14.999, 30.333]
1    고객_2  22  (14.999, 30.333]
2    고객_3  25  (14.999, 30.333]
3    고객_4  29  (14.999, 30.333]
4    고객_5  31  (30.333, 45.333]
5    고객_6  33  (30.333, 45.333]
6    고객_7  37  (30.333, 45.333]
7    고객_8  44  (30.333, 45.333]
8    고객_9  48    (45.333, 68.0]
9   고객_10  52    (45.333, 68.0]
10  고객_11  55    (45.333, 68.0]
11  고객_12  68    (45.333, 68.0]

[구간별 데이터 개수 확인]
나이구간_개수형
(14.999, 30.333]    4
(30.333, 45.333]    4
(45.333, 68.0]      4
Name: count, dtype: int64
